In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/train.csv')  # Columns: e.g., 'filename', 'label'
test_df = pd.read_csv('/kaggle/input/eurosat-dataset/EuroSAT/test.csv')    # Columns: e.g., 'filename', 'label'

classes = train_df['Label'].unique()
train_subset_df = pd.DataFrame()      

for label in classes:
    class_df = train_df[train_df['Label'] == label]
    sampled = class_df.sample(n=20, random_state=42) 
    train_subset_df = pd.concat([train_subset_df, sampled])

image_dir = '/kaggle/input/eurosat-dataset/EuroSAT/'  
train_subset_df['image_path'] = image_dir + train_subset_df['Filename']
test_df['image_path'] = image_dir + test_df['Filename']

print(f"Training subset shape: {train_subset_df.shape}")
print(f"Test set shape: {test_df.shape}")


Training subset shape: (200, 5)
Test set shape: (2700, 5)


In [2]:
import torch
print(torch.cuda.is_available())  # Should print True
print(torch.cuda.get_device_name(0))  # Prints GPU name, e.g., "NVIDIA GeForce RTX 3080"


True
Tesla P100-PCIE-16GB


In [3]:
from torch.utils.data import Dataset
from PIL import Image

class SimpleImageDataset(Dataset):
    def __init__(self, df, image_processor):
        self.image_paths = df['image_path'].tolist()
        # Ensure your DataFrame's label column is also named 'Label'
        self.labels = df['Label'].tolist()
        self.processor = image_processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Return the raw PIL image and label
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        return {"image": image, "label": label}


In [4]:
def create_collate_fn(processor):
    def collate_fn(batch):
        # 'batch' is a list of dictionaries: [{'image': img1, 'label': 1}, {'image': img2, 'label': 5}, ...]
        
        # Extract the images and labels from the list of dictionaries
        images = [item['image'] for item in batch]
        labels = [item['label'] for item in batch]
        
        # Use the processor to create the 4D tensor batch. It handles everything.
        processed_batch = processor(images=images, return_tensors="pt")
        
        # Add the labels to the batch
        processed_batch['label'] = torch.tensor(labels)
        
        return processed_batch
    return collate_fn

In [5]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from transformers import ViTImageProcessor

# --- SETUP (Do this once) ---
model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)

# 1. Create the datasets
# Use your actual dataframes here
train_ds_manual = SimpleImageDataset(train_subset_df, processor)
test_ds_manual = SimpleImageDataset(test_df, processor)

# 2. Create the collate function
my_collate_fn = create_collate_fn(processor)

# 3. Create the DataLoaders with the collate function
# This DataLoader now produces perfectly formed batches
train_dataloader_manual = DataLoader(train_ds_manual, batch_size=8, collate_fn=my_collate_fn)
test_dataloader_manual = DataLoader(test_ds_manual, batch_size=8, collate_fn=my_collate_fn)


# --- REVISED get_embeddings FUNCTION ---
def get_embeddings(model, dataloader, has_labels=True):
    model.eval()
    embeddings = []
    labels_list = [] if has_labels else None

    with torch.no_grad():
        for batch in dataloader:
            # The batch is already a perfect dictionary of tensors!
            # Just move the data to the GPU.
            inputs = {
                'pixel_values': batch['pixel_values'].to('cuda')
            }
            
            outputs = model(**inputs)
            emb = outputs.logits
            embeddings.append(emb.cpu().numpy())
            
            if has_labels:
                labels_list.append(batch['label'].numpy())

    embeddings = np.vstack(embeddings)
    if has_labels:
        labels_list = np.hstack(labels_list)
        return embeddings, labels_list
    
    return embeddings

2025-10-23 19:04:19.255898: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761246259.440755      18 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761246259.493023      18 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

In [6]:
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score


In [7]:
from transformers import ViTImageProcessor, ViTForImageClassification, ViTConfig, Trainer, TrainingArguments
model = ViTForImageClassification.from_pretrained('/kaggle/input/models/pretrained_vit/pretrained_vit').to('cuda')
model.classifier = nn.Identity()

In [8]:
train_emb_pre, train_labels_pre = get_embeddings(model, train_dataloader_manual)
test_emb_pre, test_labels_pre = get_embeddings(model, test_dataloader_manual)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

# Original tuned kernel (may still hit upper bound)
tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e3))

# Optional: Tighter bounds to force a more reasonable length_scale
# tight_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 10))

gp_pre = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)
gp_pre.fit(train_emb_pre_scaled, train_labels_pre)
pred_gp_pre = gp_pre.predict(test_emb_pre_scaled)
acc_gp_pre = accuracy_score(test_labels_pre, pred_gp_pre)
print(f"GP on Scaled Pretrained ViT Embeddings - Test Accuracy: {acc_gp_pre:.4f}")

GP on Scaled Pretrained ViT Embeddings - Test Accuracy: 0.1111


In [10]:
from sklearn.linear_model import LogisticRegression

# 1. Scale the data (still good practice)
scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

# 2. Train a simple, powerful linear model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(train_emb_pre_scaled, train_labels_pre)

# 3. Evaluate
accuracy_log_reg_pretrained = log_reg.score(test_emb_pre_scaled, test_labels_pre)
print(f"Logistic Regression on Pretrained Embeddings - Test Accuracy: {accuracy_log_reg_pretrained:.4f}")

Logistic Regression on Pretrained Embeddings - Test Accuracy: 0.8526


In [11]:
from sklearn.gaussian_process.kernels import DotProduct

linear_kernel = DotProduct(sigma_0=1.0, sigma_0_bounds="fixed")

gp_linear = GaussianProcessClassifier(kernel=linear_kernel, random_state=42, n_jobs=-1)
gp_linear.fit(train_emb_pre_scaled, train_labels_pre)

acc_gp_linear = gp_linear.score(test_emb_pre_scaled, test_labels_pre)
print(f"\nGP with Linear Kernel on Pretrained Embeddings - Test Accuracy: {acc_gp_linear:.4f}")


GP with Linear Kernel on Pretrained Embeddings - Test Accuracy: 0.8352


In [12]:
from sklearn.decomposition import PCA

scaler = StandardScaler()
train_emb_pre_scaled = scaler.fit_transform(train_emb_pre)
test_emb_pre_scaled = scaler.transform(test_emb_pre)

pca = PCA(n_components=64, random_state=42)  # Or try 50, 100
train_emb_pca = pca.fit_transform(train_emb_pre_scaled)
test_emb_pca = pca.transform(test_emb_pre_scaled)

tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 10))  # Tighter bounds
gp_pre = GaussianProcessClassifier(kernel=tuned_kernel, random_state=42, n_jobs=-1)
gp_pre.fit(train_emb_pca, train_labels_pre)
pred_gp_pre = gp_pre.predict(test_emb_pca)
acc_gp_pre = accuracy_score(test_labels_pre, pred_gp_pre)
print(f"GP on PCA-Reduced Scaled Pretrained Embeddings - Test Accuracy: {acc_gp_pre:.4f}")

GP on PCA-Reduced Scaled Pretrained Embeddings - Test Accuracy: 0.1270


In [13]:
class ViTWithGPHead:
    def __init__(self, vit_model_path, kernel, num_labels, use_pca=False, pca_components=64):
        # Load the ViT model (from saved or pretrained)
        self.vit = ViTForImageClassification.from_pretrained(vit_model_path, num_labels=num_labels, ignore_mismatched_sizes=True).to('cuda')
        self.vit.classifier = nn.Identity()  # Bypass classifier to get 768-dim embeddings
        # Freeze the backbone
        for param in self.vit.parameters():
            param.requires_grad = False
        self.gp = GaussianProcessClassifier(kernel=kernel, random_state=42, n_jobs=-1)
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=pca_components, random_state=42) if use_pca else None
        self.use_pca = use_pca

    def fit(self, train_dataloader):
        # Extract train embeddings
        self.vit.eval()
        embeddings = []
        labels_list = []
        with torch.no_grad():
            for batch in train_dataloader:
                inputs = {'pixel_values': batch['pixel_values'].to('cuda')}
                outputs = self.vit(**inputs)
                emb = outputs.logits.cpu().numpy()  # 768-dim embeddings
                embeddings.append(emb)
                labels_list.append(batch['label'].numpy())
        train_emb = np.vstack(embeddings)
        train_labels = np.hstack(labels_list)
        
        # Scale
        train_emb_scaled = self.scaler.fit_transform(train_emb)
        
        # Optional PCA (for RBF kernel in high dims)
        if self.use_pca:
            train_emb_scaled = self.pca.fit_transform(train_emb_scaled)
        
        # Fit GP head
        self.gp.fit(train_emb_scaled, train_labels)
        print("GP head fitted successfully.")

    def predict(self, test_dataloader):
        # Extract test embeddings
        self.vit.eval()
        embeddings = []
        labels_list = []
        with torch.no_grad():
            for batch in test_dataloader:
                inputs = {'pixel_values': batch['pixel_values'].to('cuda')}
                outputs = self.vit(**inputs)
                emb = outputs.logits.cpu().numpy()
                embeddings.append(emb)
                labels_list.append(batch['label'].numpy())
        test_emb = np.vstack(embeddings)
        test_labels = np.hstack(labels_list)
        
        # Scale and optional PCA
        test_emb_scaled = self.scaler.transform(test_emb)
        if self.use_pca:
            test_emb_scaled = self.pca.transform(test_emb_scaled)
        
        # Predict with GP
        preds = self.gp.predict(test_emb_scaled)
        probs = self.gp.predict_proba(test_emb_scaled)  # Optional: Get probabilities/uncertainties
        return preds, test_labels, probs

In [14]:
!pip install gpytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.9/279.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.3/176.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [15]:
# RBF kernel (use PCA to avoid underfitting in 768 dims)
tuned_kernel = 1.0 * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 10))  # Tighter bounds
vit_gp_pre_rbf = ViTWithGPHead('/kaggle/input/models/pretrained_vit/pretrained_vit', tuned_kernel, len(classes), use_pca=True, pca_components=5)
vit_gp_pre_rbf.fit(train_dataloader_manual)
preds_rbf, test_labels_rbf, probs_rbf = vit_gp_pre_rbf.predict(test_dataloader_manual)
acc_rbf = accuracy_score(test_labels_rbf, preds_rbf)
print(f"GP (RBF) Head on Frozen Pretrained ViT - Test Accuracy (PCA with 5 components): {acc_rbf:.4f}")

linear_kernel = DotProduct(sigma_0=1.0, sigma_0_bounds="fixed")

for i in [5, 10, 64, 100, 200]:
    vit_gp_pre_linear = ViTWithGPHead('/kaggle/input/models/pretrained_vit/pretrained_vit', linear_kernel, len(classes), use_pca = True, pca_components=64)
    vit_gp_pre_linear.fit(train_dataloader_manual)
    preds_linear, test_labels_linear, probs_linear = vit_gp_pre_linear.predict(test_dataloader_manual)
    acc_linear = accuracy_score(test_labels_linear, preds_linear)
    print(f"GP (Linear) Head on Frozen Pretrained ViT, PCA with {i} components - Test Accuracy: {acc_linear:.4f}")

/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning:

GP head fitted successfully.
GP (RBF) Head on Frozen Pretrained ViT - Test Accuracy (PCA with 5 components): 0.6767
GP head fitted successfully.
GP (Linear) Head on Frozen Pretrained ViT, PCA with 5 components - Test Accuracy: 0.8393
GP head fitted successfully.
GP (Linear) Head on Frozen Pretrained ViT, PCA with 10 components - Test Accuracy: 0.8393
GP head fitted successfully.
GP (Linear) Head on Frozen Pretrained ViT, PCA with 64 components - Test Accuracy: 0.8393
GP head fitted successfully.
GP (Linear) Head on Frozen Pretrained ViT, PCA with 100 components - Test Accuracy: 0.8393
GP head fitted successfully.
GP (Linear) Head on Frozen Pretrained ViT, PCA with 200 components - Test Accuracy: 0.8393


In [16]:
import gpytorch
from gpytorch.models import ApproximateGP
from gpytorch.variational import CholeskyVariationalDistribution, MultitaskVariationalStrategy, VariationalStrategy
from gpytorch.mlls import VariationalELBO
import torch
import torch.nn as nn

class MultitaskGPHead(ApproximateGP):
    def __init__(self, num_inducing, feature_dim, num_classes, kernel='rbf'):
        inducing_points = torch.randn(num_inducing, feature_dim).to('cuda')
        base_variational_distribution = gpytorch.variational.NaturalVariationalDistribution(num_inducing)
        base_variational_strategy = VariationalStrategy(
            self, inducing_points, base_variational_distribution, learn_inducing_locations=True
        )
        variational_strategy = MultitaskVariationalStrategy(base_variational_strategy, num_tasks=num_classes)
        super().__init__(variational_strategy)
        
        batch_shape = torch.Size([num_classes])
        
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=batch_shape)
        
        if kernel == 'rbf':
            self.covar_module = gpytorch.kernels.ScaleKernel(
                gpytorch.kernels.RBFKernel(ard_num_dims=feature_dim, batch_shape=batch_shape),
                batch_shape=batch_shape
            )
        # Add more kernels as needed

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

# Attach to frozen ViT (keep your code)
model = ViTForImageClassification.from_pretrained('/kaggle/input/models/pretrained_vit/pretrained_vit').to('cuda')
model.classifier = nn.Identity()  # Get embeddings
for param in model.parameters():
    param.requires_grad = False

# Extract train embeddings once (use your get_embeddings)
train_emb_pre, train_labels_pre = get_embeddings(model, train_dataloader_manual)
train_emb_tensor = torch.tensor(train_emb_pre).to('cuda')
train_labels_tensor = torch.tensor(train_labels_pre).to('cuda')  # Should be long tensor (class indices 0-9)

# Init GP head
gp_head = MultitaskGPHead(num_inducing=len(train_emb_tensor), feature_dim=768, num_classes=len(classes), kernel='rbf').to('cuda')

# Train GP (variational approximation for scalability)
likelihood = gpytorch.likelihoods.SoftmaxLikelihood(num_features=len(classes), num_classes=len(classes))
likelihood = likelihood.to('cuda')  # Add this line to fix the device mismatch
optimizer = torch.optim.Adam(gp_head.parameters(), lr=0.01)
mll = VariationalELBO(likelihood, gp_head, num_data=len(train_emb_tensor))

for epoch in range(50):  # Tune epochs
    gp_head.train()
    likelihood.train()
    optimizer.zero_grad()
    output = gp_head(train_emb_tensor)
    loss = -mll(output, train_labels_tensor)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}: Loss {loss.item():.4f}")


Epoch 0: Loss 2.3321
Epoch 1: Loss 2.3196
Epoch 2: Loss 2.3274
Epoch 3: Loss 2.3319
Epoch 4: Loss 2.3245
Epoch 5: Loss 2.3366
Epoch 6: Loss 2.3240
Epoch 7: Loss 2.3271
Epoch 8: Loss 2.3262
Epoch 9: Loss 2.3175
Epoch 10: Loss 2.3241
Epoch 11: Loss 2.3317
Epoch 12: Loss 2.3295
Epoch 13: Loss 2.3210
Epoch 14: Loss 2.3257
Epoch 15: Loss 2.3281
Epoch 16: Loss 2.3243
Epoch 17: Loss 2.3334
Epoch 18: Loss 2.3365
Epoch 19: Loss 2.3287
Epoch 20: Loss 2.3288
Epoch 21: Loss 2.3228
Epoch 22: Loss 2.3314
Epoch 23: Loss 2.3217
Epoch 24: Loss 2.3239
Epoch 25: Loss 2.3204
Epoch 26: Loss 2.3263
Epoch 27: Loss 2.3189
Epoch 28: Loss 2.3233
Epoch 29: Loss 2.3240
Epoch 30: Loss 2.3268
Epoch 31: Loss 2.3263
Epoch 32: Loss 2.3280
Epoch 33: Loss 2.3195
Epoch 34: Loss 2.3245
Epoch 35: Loss 2.3238
Epoch 36: Loss 2.3148
Epoch 37: Loss 2.3232
Epoch 38: Loss 2.3214
Epoch 39: Loss 2.3208
Epoch 40: Loss 2.3190
Epoch 41: Loss 2.3185
Epoch 42: Loss 2.3292
Epoch 43: Loss 2.3278
Epoch 44: Loss 2.3313
Epoch 45: Loss 2.319

In [17]:
test_emb_pre, test_labels_pre = get_embeddings(model, test_dataloader_manual)
test_emb_tensor = torch.tensor(test_emb_pre).to('cuda')
gp_head.eval()
likelihood.eval()
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    preds_dist = likelihood(gp_head(test_emb_tensor))
    preds = preds_dist.mean.argmax(dim=0).cpu().numpy()  # Fix: argmax(dim=0) over classes
acc = accuracy_score(test_labels_pre, preds)
print(f"GPyTorch GP Head Accuracy: {acc:.4f}")

GPyTorch GP Head Accuracy: 0.1111
